# Experiment 4 — Comparative Study of Deep CNN Architectures Using Transfer Learning
**CS3807 – Deep Learning Laboratory**

Model used for transfer learning: **VGG16** (pretrained on ImageNet)
Dataset: **CIFAR-10**

Run cells top to bottom in Google Colab (enable GPU: Runtime → Change runtime type → T4 GPU).


## Task 1: Dataset Preparation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.utils import to_categorical

(x_train, y_train), (x_test, y_test) = cifar10.load_data()

class_names = ['Airplane','Automobile','Bird','Cat','Deer',
               'Dog','Frog','Horse','Ship','Truck']

# Normalize pixel values to [0, 1]
x_train = x_train.astype('float32') / 255.0
x_test  = x_test.astype('float32') / 255.0

# One-hot encode labels
y_train_cat = to_categorical(y_train, 10)
y_test_cat  = to_categorical(y_test, 10)

print("Training data shape:", x_train.shape)
print("Training labels shape:", y_train_cat.shape)
print("Testing data shape:", x_test.shape)
print("Testing labels shape:", y_test_cat.shape)


In [ ]:
# Display 10 sample images
plt.figure(figsize=(12, 5))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(x_train[i])
    plt.title(class_names[y_train[i][0]])
    plt.axis('off')
plt.suptitle("Sample CIFAR-10 Images")
plt.tight_layout()
plt.savefig("sample_cifar10_images.png", dpi=150)
plt.show()


**Inference:** CIFAR-10 images are 32x32x3 low-resolution color images spanning 10 balanced classes, which makes transfer learning from ImageNet-pretrained networks attractive since these models already know generic edge/texture/shape features.

## Task 2: Transfer Learning Setup (VGG16)

Steps: load pretrained ImageNet weights -> remove original classifier -> freeze conv base ->
add GlobalAveragePooling -> Dense(ReLU) -> Dense(Softmax).

Note: VGG16 expects inputs of at least 32x32 (we use CIFAR-10's native 32x32) and 3 channels, which matches directly.


In [ ]:
from tensorflow.keras.applications import VGG16
from tensorflow.keras import layers, models
from tensorflow.keras.applications.vgg16 import preprocess_input

IMG_SHAPE = (32, 32, 3)

# 1. Load pretrained model (ImageNet weights), without its classifier head
base_model = VGG16(weights='imagenet', include_top=False, input_shape=IMG_SHAPE)

# 2 & 3. Freeze the convolutional base
base_model.trainable = False

# 4-6. Add Global Average Pooling + Dense(ReLU) + Dense(Softmax)
inputs = tf.keras.Input(shape=IMG_SHAPE)
x = preprocess_input(inputs * 255.0)   # VGG16 preprocessing expects 0-255 range
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(256, activation='relu')(x)
outputs = layers.Dense(10, activation='softmax')(x)

model = models.Model(inputs, outputs)

model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

model.summary()


## Task 3: Model Training (frozen base)

In [ ]:
history = model.fit(
    x_train, y_train_cat,
    validation_data=(x_test, y_test_cat),
    batch_size=32,
    epochs=15,
    verbose=1
)


In [ ]:
# Training/Validation Accuracy and Loss plots
plt.figure(figsize=(6,4))
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.xlabel('Epoch'); plt.ylabel('Accuracy'); plt.title('Training Accuracy'); plt.legend()
plt.savefig("training_accuracy.png", dpi=150); plt.show()

plt.figure(figsize=(6,4))
plt.plot(history.history['val_accuracy'], label='Validation Accuracy', color='orange')
plt.xlabel('Epoch'); plt.ylabel('Accuracy'); plt.title('Validation Accuracy'); plt.legend()
plt.savefig("validation_accuracy.png", dpi=150); plt.show()

plt.figure(figsize=(6,4))
plt.plot(history.history['loss'], label='Training Loss')
plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.title('Training Loss'); plt.legend()
plt.savefig("training_loss.png", dpi=150); plt.show()

plt.figure(figsize=(6,4))
plt.plot(history.history['val_loss'], label='Validation Loss', color='orange')
plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.title('Validation Loss'); plt.legend()
plt.savefig("validation_loss.png", dpi=150); plt.show()


In [ ]:
pre_ft_test_loss, pre_ft_test_acc = model.evaluate(x_test, y_test_cat, verbose=0)
print("Test accuracy BEFORE fine-tuning: {:.4f}".format(pre_ft_test_acc))


## Task 4: Fine-Tuning (unfreeze last conv block)

In [ ]:
# Unfreeze only the last convolutional block of VGG16 (block5)
base_model.trainable = True
for layer in base_model.layers:
    if not layer.name.startswith('block5'):
        layer.trainable = False

# Recompile with a smaller learning rate for fine-tuning
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

fine_tune_history = model.fit(
    x_train, y_train_cat,
    validation_data=(x_test, y_test_cat),
    batch_size=32,
    epochs=8,
    verbose=1
)


In [ ]:
post_ft_test_loss, post_ft_test_acc = model.evaluate(x_test, y_test_cat, verbose=0)

print("Test accuracy BEFORE fine-tuning: {:.4f}".format(pre_ft_test_acc))
print("Test accuracy AFTER  fine-tuning: {:.4f}".format(post_ft_test_acc))
print("Improvement: {:.4f}".format(post_ft_test_acc - pre_ft_test_acc))


## Task 5: Model Evaluation

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, precision_score, recall_score, f1_score
import seaborn as sns

y_pred_probs = model.predict(x_test)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = y_test.flatten()

acc = (y_pred == y_true).mean()
prec = precision_score(y_true, y_pred, average='macro')
rec = recall_score(y_true, y_pred, average='macro')
f1 = f1_score(y_true, y_pred, average='macro')

print(f"Accuracy : {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall   : {rec:.4f}")
print(f"F1-score : {f1:.4f}")

print("\nClassification Report:\n")
print(classification_report(y_true, y_pred, target_names=class_names))


In [ ]:
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted'); plt.ylabel('Actual'); plt.title('Confusion Matrix')
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=150)
plt.show()


In [ ]:
# Optional: a few misclassified images
misclassified_idx = np.where(y_pred != y_true)[0][:10]

plt.figure(figsize=(12,5))
for i, idx in enumerate(misclassified_idx):
    plt.subplot(2,5,i+1)
    plt.imshow(x_test[idx])
    plt.title(f"T:{class_names[y_true[idx]]}\nP:{class_names[y_pred[idx]]}", fontsize=8)
    plt.axis('off')
plt.suptitle("Misclassified Images")
plt.tight_layout()
plt.savefig("misclassified_images.png", dpi=150)
plt.show()


## Results Table (fill in after running)

| Metric | Value |
|---|---|
| Training Accuracy | |
| Testing Accuracy | |
| Precision | |
| Recall | |
| F1-score | |
| Training Time | |
| Total Parameters | |

## Discussion Questions (answer briefly in your own words)

1. Why is AlexNet considered a breakthrough in deep learning?
2. Why does VGG16 use only 3x3 convolution filters?
3. Explain the advantages of the Inception module.
4. What is the purpose of residual learning?
5. Differentiate LeNet and ResNet.
6. What is Transfer Learning?
7. Why is fine tuning required?
8. Explain the difference between dilated convolution and transpose convolution.
9. Why do pretrained models converge faster?
10. Compare the computational complexity of LeNet and ResNet.
